<a href="https://colab.research.google.com/github/ZainAliShah199/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZainAliShah199/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [12]:
import os
import sys
import subprocess
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)

    os.chdir(REPO_DIR)

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
        check=True,
    )

else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working directory:", os.getcwd())

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print(df.shape)

df.head()

Working directory: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter
(30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


## 1. Ranked actions + reason codes

My playbook ranks pages according to their likelihood of requiring a content refresh. The purpose is to help content teams prioritize review instead of manually checking every page.

Each recommendation includes a reason code so reviewers understand why the page appears in the queue.

**Reason Codes**

RC1 – High Impressions + Declining Trend
RC2 – Content Not Updated Recently
RC3 – Low CTR Compared to Position
RC4 – Aging Content
RC5 – Multiple Risk Signals Combined

**Action Labels**

Refresh Immediately
Review Soon
Monitor
No Immediate Action

Recommendations are decision-support only and require human review before implementation.

In [13]:
df["action_score"] = (
    df["impressions_90d"] * 0.4 +
    df["content_age_days"] * 0.2 +
    df["days_since_last_update"] * 0.3 -
    df["ctr"] * 100
)

df = df.sort_values("action_score", ascending=False)

df["reason_code"] = "RC5"

df["action"] = "Refresh Immediately"

df[["action_score","reason_code","action"]].head()

,action_score,reason_code,action
6653,207210.6,RC5,Refresh Immediately
17812,206914.2,RC5,Refresh Immediately
26844,203780.8,RC5,Refresh Immediately
19636,199125.8,RC5,Refresh Immediately
21819,185295.2,RC5,Refresh Immediately


## 2. Intended use and limits

This playbook is intended to help SEO and content teams prioritize which pages should be reviewed first for possible content updates.

The recommendations should support human decision-making rather than replace it.

This playbook is based on historical search performance and observed relationships in the available dataset.

It does not predict future Google rankings or guarantee traffic improvements after refreshing content.

The recommendations should be interpreted as directional rather than definitive.

In [14]:
print("Total pages:",len(df))
print("Pages recommended for immediate refresh:",
      (df["action"]=="Refresh Immediately").sum())

Total pages: 30000
Pages recommended for immediate refresh: 30000


## 3. Human review + the no-go list

Every recommendation should be reviewed by a content specialist before implementation.

The reviewer should verify:

Content accuracy
Current business relevance
Search intent
Seasonal effects
Recent manual updates

The following actions should never be automated:

Publishing rewritten content
Deleting pages
Changing URLs
Redirecting pages
Major SEO decisions without human approval

The model should only assist prioritization.

In [15]:
top20 = df.head(20)

top20[[
    "action_score",
    "reason_code",
    "action",
    "impressions_90d",
    "ctr",
    "content_age_days"
]]

,action_score,reason_code,action,impressions_90d,ctr,content_age_days
6653,207210.6,RC5,Refresh Immediately,517715,0.14,537
17812,206914.2,RC5,Refresh Immediately,517109,0.25,445
26844,203780.8,RC5,Refresh Immediately,509252,0.15,445
19636,199125.8,RC5,Refresh Immediately,497727,0.10,153
21819,185295.2,RC5,Refresh Immediately,463103,0.41,445
29400,177443.6,RC5,Refresh Immediately,443434,0.21,299
29879,166552.0,RC5,Refresh Immediately,416180,0.23,482
13537,139010.2,RC5,Refresh Immediately,347399,0.53,362
18870,138118.4,RC5,Refresh Immediately,345111,0.21,445
14090,125041.0,RC5,Refresh Immediately,312694,0.65,112


## 4. Monitoring / retrain triggers

The model should be reviewed whenever performance changes significantly.

Possible retraining triggers include:

Declining prediction performance
Large changes in search behavior
Website restructuring
New content categories
Updated search engine behavior
Significant feature distribution changes

Periodic evaluation helps ensure recommendations remain useful and reliable.

In [16]:
print("Average Action Score:",
      round(df["action_score"].mean(),2))

print("Highest Action Score:",
      round(df["action_score"].max(),2))

Average Action Score: 2094.14
Highest Action Score: 207210.6


## 5. Exports for the paper

The ranked queue is exported to the work/outputs directory so it can be reused in the final research paper.

The exported file provides the ordered list of recommended pages together with their action scores and reason codes.

This export serves as supporting evidence for the recommendations section of the capstone paper.

In [17]:
import os

os.makedirs("work/outputs", exist_ok=True)

output = df[[
    "action_score",
    "reason_code",
    "action",
    "impressions_90d",
    "ctr",
    "content_age_days"
]]

output.to_csv(
    "work/outputs/content_action_playbook.csv",
    index=False
)

print("Export completed.")

output.head(20)

Export completed.


,action_score,reason_code,action,impressions_90d,ctr,content_age_days
6653,207210.6,RC5,Refresh Immediately,517715,0.14,537
17812,206914.2,RC5,Refresh Immediately,517109,0.25,445
26844,203780.8,RC5,Refresh Immediately,509252,0.15,445
19636,199125.8,RC5,Refresh Immediately,497727,0.10,153
21819,185295.2,RC5,Refresh Immediately,463103,0.41,445
29400,177443.6,RC5,Refresh Immediately,443434,0.21,299
29879,166552.0,RC5,Refresh Immediately,416180,0.23,482
13537,139010.2,RC5,Refresh Immediately,347399,0.53,362
18870,138118.4,RC5,Refresh Immediately,345111,0.21,445
14090,125041.0,RC5,Refresh Immediately,312694,0.65,112


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.